# Cat Image Dataset Exploration

Explores the `huggingface/cats-image` dataset using the `datasets` library. The notebook reads `DATA_ROOT` from the workspace `.env` (the canonical place for shared data-volume paths) and points the Hugging Face cache at `${DATA_ROOT}/huggingface` so downloads land on the configured external volume rather than the user's home directory.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Walk up from the notebook to find the workspace .env (alongside the top-level
# CLAUDE.md / AGENTS.md). Loading it before importing `datasets` is required —
# huggingface_hub reads HF_HOME at import time of its cache module.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / ".env").exists():
        load_dotenv(candidate / ".env")
        break

# DATA_ROOT chooses WHERE the HF cache lives; it is not required to run this
# notebook. On a workstation it points the cache at the configured volume so a
# large dataset does not land in the home directory. Where it is unset — a CI
# runner, or a fresh clone with no .env — huggingface_hub uses its own default
# cache and the notebook runs unchanged. A DATA_ROOT that is SET but missing is
# still an error: that is a misconfigured machine, not an absent preference.
data_root = os.getenv("DATA_ROOT")
if data_root:
    if not Path(data_root).exists():
        raise RuntimeError(f"DATA_ROOT={data_root!r} does not exist on this machine — mount the volume or update .env.")
    # Point HF cache at the configured volume regardless of any inherited HF_HOME.
    os.environ["HF_HOME"] = str(Path(data_root) / "huggingface")

print(f"DATA_ROOT = {data_root or '(unset — using the default HF cache)'}")
print(f"HF_HOME   = {os.environ.get('HF_HOME', '(default)')}")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("huggingface/cats-image", split="test")

print(f"Dataset loaded with {len(dataset)} examples.")
print(f"Features: {dataset.features}")

In [ ]:
import matplotlib.pyplot as plt

image = dataset[0]["image"]
plt.figure(figsize=(10, 8))
plt.imshow(image)
plt.axis("off")
plt.title("Hugging Face Cat Image")
plt.show()